# EV01 — Define the prediction moment
ML Mastery — Learn Machine Learning by Doing. Trusted author code, not learner-hosted execution.

This feasibility section belongs to the eventual EV notebook. No estimator is fitted; no credentials, uploads, external data or platform evidence writes. Genuine Colab execution remains unverified. Original fictional rows are not a production documentation-quality dataset.

In [ ]:
import sys, json, random
from importlib.metadata import version
assert sys.version_info[:3] == (3, 13, 16), 'Use the approved local Python profile, or separately verify a new profile.'
pins = {'numpy': '2.5.4', 'pandas': '3.0.6', 'matplotlib': '3.11.2', 'scikit-learn': '1.9.1'}
for package, expected in pins.items():
    assert version(package) == expected, f'Unsupported {package} profile'
random.seed(0)
print('Approved local profile asserted; seed=0')

## Problem and visual boundary
At t=0, predict a future house price, cancellation during days1–30, or a human-reviewed label for a submitted guide (Good=0, Needs Improvement=1).

Before t=0: known inputs → t=0: prediction → after t=0: outcome/review. Later outcomes can become training labels, not current features. A correlated post-review flag is unavailable at submission. Predict before running: is an available feature plus a row-overlapping split valid?

In [ ]:
def scenarios():
    specs = [
        ("housing-units", "House price: new independent homes", "housing", False, False),
        ("housing-future", "House price: future sales", "housing", False, True),
        ("churn-customers", "Churn: new customers", "churn", True, False),
        ("churn-future", "Churn: future events for known customers", "churn", False, True),
        ("docs-groups", "Documentation: unseen source/template groups", "docs", True, False),
        ("docs-future", "Documentation: future revisions from unseen families", "docs", True, True),
    ]
    result = []
    for index, (identifier, title, domain, group, time) in enumerate(specs):
        rows = [{"id": f"r{position}", "group": f"g{(position - 1) // 2 + 1}" if group else (f"g{position % 2}" if identifier == "churn-future" else f"g{position}"), "time": position} for position in range(1, 7)]
        plans = [
            {"id": "holdout", "label": "Train r1–r4; test r5–r6", "train": ["r1", "r2", "r3", "r4"], "test": ["r5", "r6"]},
            {"id": "overlap", "label": "Train r1–r4; test r4–r6", "train": ["r1", "r2", "r3", "r4"], "test": ["r4", "r5", "r6"]},
            {"id": "crossed", "label": "Train r1,r3,r5; test r2,r4,r6" if group else "Train r1,r5,r6; test r2,r3,r4", "train": ["r1", "r3", "r5"] if group else ["r1", "r5", "r6"], "test": ["r2", "r4", "r6"] if group else ["r2", "r3", "r4"]},
        ]
        if not group and not time:
            plans[2] = {"id": "crossed", "label": "Train r1–r5; test r5–r6", "train": ["r1", "r2", "r3", "r4", "r5"], "test": ["r5", "r6"]}
        safe, future = {"housing": ("area at listing", "future sale proceeds"), "churn": ("usage before decision", "cancellation after decision"), "docs": ("submitted text at review request", "post-review correction flag")}[domain]
        result.append({"id": identifier, "title": title, "domain": domain, "requireGroup": group, "requireTime": time, "rows": rows, "plans": plans[index % 3:] + plans[:index % 3], "features": [{"id": "available", "label": safe, "observedAt": -1}, {"id": "future", "label": future, "observedAt": 1}]})
    return result

def analyze(scene, feature_id, plan_id):
    feature = next(item for item in scene["features"] if item["id"] == feature_id)
    plan = next(item for item in scene["plans"] if item["id"] == plan_id)
    rows = {row["id"]: row for row in scene["rows"]}
    overlap = sorted(set(plan["train"]) & set(plan["test"]))
    groups = sorted({rows[key]["group"] for key in plan["train"]} & {rows[key]["group"] for key in plan["test"]}) if scene["requireGroup"] else []
    future = [key for key in plan["train"] if rows[key]["time"] >= min(rows[item]["time"] for item in plan["test"])] if scene["requireTime"] else []
    feature_valid = feature["observedAt"] <= 0
    split_valid = not (overlap or groups or future)
    reason = "both-valid" if feature_valid and split_valid else "feature-future" if split_valid else "split-invalid" if feature_valid else "both-invalid"
    return {"featureValid": feature_valid, "splitValid": split_valid, "valid": feature_valid and split_valid, "overlap": overlap, "groups": groups, "future": future, "reason": reason}


In [ ]:
scenes = scenarios()
scene = scenes[0]
result = analyze(scene, 'available', 'overlap')
assert result['overlap'] == ['r4']
assert not result['valid']
print(json.dumps(result, sort_keys=True))

## Mathematics, Python and changed conditions
E(0)={j: observed_at[j]≤0}; T∩H must be empty. Unseen entities additionally require disjoint groups, and future-cohort evaluation requires earlier training times. These necessary conditions do not replace a real label-availability audit.

Guided repair: replace the authored plan ID `overlap` with `holdout`, not arbitrary code. Then change `available` to `future`: a good split does not repair an unavailable feature.

In [ ]:
safe = analyze(scene, 'available', 'holdout')
future_feature = analyze(scene, 'future', 'holdout')
assert safe['valid']
assert not future_feature['valid'] and future_feature['splitValid']
print(json.dumps({'safe': safe['valid'], 'future_feature': future_feature['reason']}, sort_keys=True))

In [ ]:
results = []
for scene in scenes:
    for feature in scene['features']:
        for plan in scene['plans']:
            results.append({'config': scene['id'] + ':' + feature['id'] + ':' + plan['id'], **analyze(scene, feature['id'], plan['id'])})
assert len(results) == 36
assert sum(result['valid'] for result in results) == 6
assert analyze(scenes[2], 'available', 'crossed')['groups'] == ['g1', 'g2', 'g3']
assert analyze(scenes[1], 'available', 'crossed')['future'] == ['r5', 'r6']
assert analyze(scenes[5], 'available', 'crossed')['future'] == ['r3', 'r5']
print(json.dumps({'states': len(results), 'valid': sum(result['valid'] for result in results), 'results': results}, sort_keys=True))

## Evaluate and compare
All36 configurations check information permissions, not fitted performance. Linear Regression produces a numeric price; classification models are candidates for churn/document labels, but no algorithm repairs leakage. Association does not establish that increasing usage or padding words improves an outcome.

Independent transfer: explain why a disjoint random row split might still leak documentation templates; contrast future events of known customers with unseen-customer deployment. No notebook execution grants mastery.

Expected assertions: r4 overlaps; six valid states out of36; group overlap g1/g2/g3; future-training rows r5/r6 for housing and r3/r5 for future documentation. Local two-kernel reproducibility is distinct from genuine Colab acceptance.